In [2]:
#| include: false
#Bloc 1 : remplace les deux anciens blocs de chargement
#Ce que fait chaque partie
#1.1 recopie les tableaux des quatre pages du site. Les numéros ont été vérifiés aujourd’hui sur les pages elles-mêmes, et None correspond aux astérisques (données non disponibles).
#1.2 transforme ces tableaux en une table, le catalogue, avec une ligne par fichier et un nom qui résume tout. Par exemple, 12kDE_OR021at3_L2 se lit : 12 kHz, défaut côté entraînement, bague extérieure, 0,021", position 3 h, charge 2 HP.
#1.3 définit charger(nom). Elle télécharge le fichier la première fois, puis le relit en local. Elle choisit la bonne variable quand un fichier en contient deux, et prend la vitesse nominale si la vitesse mesurée manque. Ce sont les deux défauts du code précédent.

#Résultat attendu : 161 fichiers au catalogue, puis un tableau de synthèse :
import io
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal as sci
from scipy.io import loadmat
from scipy.signal import hilbert
from scipy.stats import kurtosis
from scipy.fftpack import fft, fftfreq

# ------------------------------------------------------------------
# 1.1 Les numéros de fichiers du site, page par page
#     clé = (défaut, diamètre en millièmes de pouce)
#     valeur = numéros pour les charges 0, 1, 2, 3 HP (None = absent)
# ------------------------------------------------------------------
NORMAL = [97, 98, 99, 100]

PAGES = {
    "12kDE": (12000, {
        ("IR", 7): [105, 106, 107, 108],      ("B", 7): [118, 119, 120, 121],
        ("OR@6", 7): [130, 131, 132, 133],    ("OR@3", 7): [144, 145, 146, 147],
        ("OR@12", 7): [156, 158, 159, 160],
        ("IR", 14): [169, 170, 171, 172],     ("B", 14): [185, 186, 187, 188],
        ("OR@6", 14): [197, 198, 199, 200],
        ("IR", 21): [209, 210, 211, 212],     ("B", 21): [222, 223, 224, 225],
        ("OR@6", 21): [234, 235, 236, 237],   ("OR@3", 21): [246, 247, 248, 249],
        ("OR@12", 21): [258, 259, 260, 261],
        ("IR", 28): [3001, 3002, 3003, 3004], ("B", 28): [3005, 3006, 3007, 3008],
    }),
    "48kDE": (48000, {
        ("IR", 7): [109, 110, 111, 112],      ("B", 7): [122, 123, 124, 125],
        ("OR@6", 7): [135, 136, 137, 138],    ("OR@3", 7): [148, 149, 150, 151],
        ("OR@12", 7): [161, 162, 163, 164],
        ("IR", 14): [174, 175, 176, 177],     ("B", 14): [189, 190, 191, 192],
        ("OR@6", 14): [201, 202, 203, 204],
        ("IR", 21): [213, 214, 215, 217],     ("B", 21): [226, 227, 228, 229],
        ("OR@6", 21): [238, 239, 240, 241],   ("OR@3", 21): [250, 251, 252, 253],
        ("OR@12", 21): [262, 263, 264, 265],
    }),
    "12kFE": (12000, {
        ("IR", 7): [278, 279, 280, 281],      ("B", 7): [282, 283, 284, 285],
        ("OR@6", 7): [294, 295, 296, 297],    ("OR@3", 7): [298, 299, 300, 301],
        ("OR@12", 7): [302, 305, 306, 307],
        ("IR", 14): [274, 275, 276, 277],     ("B", 14): [286, 287, 288, 289],
        ("OR@6", 14): [313, None, None, None], ("OR@3", 14): [310, 309, 311, 312],
        ("IR", 21): [270, 271, 272, 273],     ("B", 21): [290, 291, 292, 293],
        ("OR@6", 21): [315, None, None, None], ("OR@3", 21): [None, 316, 317, 318],
    }),
}

RPM_NOMINAL = {0: 1797, 1: 1772, 2: 1750, 3: 1730}

# ------------------------------------------------------------------
# 1.2 Le catalogue : une ligne par fichier, avec un nom lisible
#     ex. 'Normal_L0', '12kDE_IR007_L0', '12kDE_OR021at3_L2', '48kDE_B014_L1'
# ------------------------------------------------------------------
lignes = []
for charge, num in enumerate(NORMAL):
    lignes.append({"nom": f"Normal_L{charge}", "numero": num, "source": "Normal",
                   "fs": 12000, "defaut": "Sain", "position_OR": None,
                   "diametre_mils": 0, "charge_hp": charge})

for source, (fs, table) in PAGES.items():
    for (defaut, diametre), numeros in table.items():
        localisation, _, position = defaut.partition("@")
        for charge, num in enumerate(numeros):
            if num is None:
                continue
            nom = f"{source}_{localisation}{diametre:03d}" + (f"at{position}" if position else "") + f"_L{charge}"
            lignes.append({"nom": nom, "numero": num, "source": source, "fs": fs,
                           "defaut": localisation, "position_OR": position or None,
                           "diametre_mils": diametre, "charge_hp": charge})

catalogue = pd.DataFrame(lignes).set_index("nom")
catalogue["rpm_nominal"] = catalogue["charge_hp"].map(RPM_NOMINAL)

# ------------------------------------------------------------------
# 1.3 Lecture d'un fichier depuis le site (copie locale sous un nom lisible)
# ------------------------------------------------------------------
URL_CWRU = "https://engineering.case.edu/sites/default/files/"
CACHE = Path("../data/cwru_cache")

def charger(nom):
    """Renvoie {'DE', 'FE', 'BA', 'fs', 'rpm'} pour un fichier du catalogue
    (ex. '12kDE_IR007_L0')."""
    info = catalogue.loc[nom]
    numero = int(info["numero"])
    chemin = CACHE / f"{nom}__{numero}.mat"            # ex. 12kDE_IR007_L0__105.mat

    if not chemin.exists():                            # 1re fois : téléchargement depuis le site
        CACHE.mkdir(parents=True, exist_ok=True)
        with urllib.request.urlopen(URL_CWRU + f"{numero}.mat") as reponse:
            chemin.write_bytes(reponse.read())
    mat = loadmat(chemin)

    def choisir(suffixe):                              # privilégie la variable du bon numéro
        cles = [k for k in mat if k.endswith(suffixe)]
        bonne = [k for k in cles if k.startswith(f"X{numero:03d}")]
        return (bonne or cles or [None])[0]

    signaux = {"fs": int(info["fs"])}
    for canal in ["DE", "FE", "BA"]:
        cle = choisir(f"_{canal}_time")
        signaux[canal] = None if cle is None else np.ravel(mat[cle])
    cle_rpm = choisir("RPM")
    signaux["rpm"] = float(np.squeeze(mat[cle_rpm])) if cle_rpm else float(info["rpm_nominal"])
    return signaux

from IPython.display import Markdown

tab = catalogue.groupby(["source", "defaut"]).size().unstack(fill_value=0)
tab = tab[["Sain", "IR", "B", "OR"]].reindex(["Normal", "12kDE", "48kDE", "12kFE"])
tab = tab.reset_index().rename(columns={"source": "Source", "IR": "Bague int. (IR)",
                                        "B": "Bille (B)", "OR": "Bague ext. (OR)"})
Markdown(tab.to_markdown(index=False))

| Source   |   Sain |   Bague int. (IR) |   Bille (B) |   Bague ext. (OR) |
|:---------|-------:|------------------:|------------:|------------------:|
| Normal   |      4 |                 0 |           0 |                 0 |
| 12kDE    |      0 |                16 |          16 |                28 |
| 48kDE    |      0 |                12 |          12 |                28 |
| 12kFE    |      0 |                12 |          12 |                21 |

In [3]:
import re
from scipy.signal import butter, sosfiltfilt

TAILLE = 2400   # fenêtres de 0,2 s à 12 kHz, comme dans les rapports 1 et 2
SOS = butter(4, (2000, 5000), btype="bandpass", fs=12000, output="sos")   # bande de résonance

def infos_nom(nom):
    """Localisation, diamètre et charge, lus dans le nom du fichier."""
    if nom.startswith("Normal"):
        return "Sain", 0, int(nom[-1])
    m = re.match(r"^\w+?_([A-Z]+)(\d{3})(?:at\d+)?_L(\d)$", nom)   # ex. 12kDE_OR021at6_L2
    return m.group(1), int(m.group(2)), int(m.group(3))

def indicateurs(nom, canal):
    """RMS et RMS_bande de chaque fenêtre d'un fichier, pour un capteur."""
    x = charger(nom)[canal]
    if x is None:
        return pd.DataFrame()
    xb = sosfiltfilt(SOS, x)                       # signal filtré 2–5 kHz
    n = len(x) // TAILLE
    f = x[:n * TAILLE].reshape(n, TAILLE)
    fb = xb[:n * TAILLE].reshape(n, TAILLE)
    defaut, diametre, charge = infos_nom(nom)
    return pd.DataFrame({"fichier": nom, "canal": canal, "defaut": defaut,
                         "diametre": diametre, "charge": charge,
                         "RMS": np.sqrt((f ** 2).mean(axis=1)),
                         "RMS_bande": np.sqrt((fb ** 2).mean(axis=1))})

In [5]:
sains = [n for n in catalogue.index if n.startswith("Normal")]
de = [n for n in catalogue.index if n.startswith("12kDE_")]   # défauts côté entraînement
fe = [n for n in catalogue.index if n.startswith("12kFE_")]   # défauts côté ventilateur

liste = [(n, "DE") for n in sains + de] + [(n, "FE") for n in sains + fe]
morceaux = []
for i, (nom, canal) in enumerate(liste, 1):
    print(f"\r{i}/{len(liste)} {nom}          ", end="")
    morceaux.append(indicateurs(nom, canal))
table = pd.concat(morceaux, ignore_index=True)
print()
table.groupby(["canal", "defaut"]).size()

113/113 12kFE_OR021at3_L3          


canal  defaut
DE     B          801
       IR         801
       OR        1405
       Sain       706
FE     B          600
       IR         600
       OR        1050
       Sain       706
dtype: int64

In [6]:
def evaluer(app, test, var, q=0.99):
    seuil = app[var].quantile(q)                   # seuil fixé sur les sains d'apprentissage
    sain, defa = test[test["defaut"] == "Sain"], test[test["defaut"] != "Sain"]
    return {"seuil": seuil,
            "fausses alertes (%)": 100 * (sain[var] > seuil).mean(),
            "défauts manqués (%)": 100 * (defa[var] <= seuil).mean(),
            "marge Δ": defa[var].min() - sain[var].max()}   # > 0 : séparation parfaite

de_t, fe_t = table[table["canal"] == "DE"], table[table["canal"] == "FE"]

def sains_app(t):
    return t[(t["defaut"] == "Sain") & (t["charge"] <= 2)]

def test_3hp(t):
    return t[t["charge"] == 3]

scenarios = {"DE, seuil appris sur DE": (sains_app(de_t), test_3hp(de_t)),
             "FE, seuil appris sur DE": (sains_app(de_t), test_3hp(fe_t)),
             "FE, seuil appris sur FE": (sains_app(fe_t), test_3hp(fe_t))}

resultats = pd.DataFrame({(s, v): evaluer(a, t, v)
                          for s, (a, t) in scenarios.items()
                          for v in ["RMS", "RMS_bande"]}).T.round(3)
resultats

seuil  fausses alertes (%)  \
DE, seuil appris sur DE RMS        0.077                0.000   
                        RMS_bande  0.019              100.000   
FE, seuil appris sur DE RMS        0.077               98.020   
                        RMS_bande  0.019              100.000   
FE, seuil appris sur FE RMS        0.089                0.495   
                        RMS_bande  0.020              100.000   

                                   défauts manqués (%)  marge Δ  
DE, seuil appris sur DE RMS                        0.0    0.011  
                        RMS_bande                  0.0    0.025  
FE, seuil appris sur DE RMS                        0.0    0.015  
                        RMS_bande                  0.0    0.046  
FE, seuil appris sur FE RMS                        0.0    0.015  
                        RMS_bande                  0.0    0.046

In [7]:
t = test_3hp(de_t)
print("Sain à 3 HP, maximum :", t.loc[t["defaut"] == "Sain", ["RMS", "RMS_bande"]].max().round(3).to_dict())
t[t["defaut"] != "Sain"].groupby("fichier")[["RMS", "RMS_bande"]].min().sort_values("RMS").head(6).round(3)

Sain à 3 HP, maximum : {'RMS': 0.07, 'RMS_bande': 0.021}


,RMS,RMS_bande
fichier,,
12kDE_B014_L3,0.081,0.047
12kDE_OR014at6_L3,0.086,0.071
12kDE_B021_L3,0.113,0.091
12kDE_B007_L3,0.142,0.133
12kDE_IR014_L3,0.164,0.148
12kDE_OR007at12_L3,0.232,0.225


In [8]:
s = table[table["defaut"] == "Sain"]
s.groupby(["canal", "charge"])[["RMS", "RMS_bande"]].agg(["median", "max"]).round(3)

RMS        RMS_bande       
             median    max    median    max
canal charge                               
DE    0       0.074  0.078     0.011  0.012
      1       0.066  0.075     0.016  0.017
      2       0.064  0.068     0.019  0.020
      3       0.066  0.070     0.021  0.021
FE    0       0.084  0.093     0.014  0.015
      1       0.075  0.080     0.018  0.018
      2       0.072  0.078     0.020  0.021
      3       0.082  0.089     0.022  0.023